In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [2]:
df = pd.read_csv('/content/heart_disease_uci.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/heart_disease_uci.csv'

In [ ]:
df.head()

#### Data Dictionary

| Column | Description |
|---|---|
| `id` | Patient ID |
| `age` | Patient's age |
| `sex` | Patient's sex |
| `dataset` | Dataset or location where the data was collected |
| `cp` | Type of chest pain |
| `trestbps` | Resting blood pressure |
| `chol` | Cholesterol level |
| `fbs` | Whether fasting blood sugar is greater than 120 mg/dL |
| `restecg` | Resting electrocardiogram (ECG) result |
| `thalch` | Maximum heart rate achieved |
| `exang` | Whether exercise-induced angina occurs |
| `oldpeak` | ST depression during exercise compared with rest |
| `slope` | Slope of the ST segment during exercise |
| `ca` | Number of major vessels colored by fluoroscopy |
| `thal` | Thalassemia test result |
| `num` | Presence/severity of heart disease — target variable |

In [ ]:
df.drop('id',axis=1,inplace=True)

Before modeling, we visualize every numeric and categorical column to
understand distributions, class balance, and potential data-quality
issues (missing values are common in this dataset, especially in the
later-added hospital sources).

In [ ]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
categorical_cols = ['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'num']

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor='black')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### Numeric Feature Distributions

- **`age`**: Roughly bell-shaped, centered around 55-60, ranging from
  ~28 to ~77 — a realistic adult patient population with no obvious
  outliers.
- **`trestbps`** (resting blood pressure): Mostly concentrated between
  100-150, with a long right tail up to ~200. A few near-zero values are
  visible — likely invalid/missing data encoded as 0, which will need
  cleaning.
- **`chol`** (cholesterol): Shows a **suspicious spike at 0**, which is
  clinically impossible (cholesterol can't be zero) — this is almost
  certainly missing data encoded as 0 rather than NaN, and needs to be
  treated as missing before analysis. Excluding that spike, the real
  distribution is roughly normal, centered around 200-250.
- **`thalch`** (max heart rate achieved): Roughly normal, centered around
  140-150, consistent with expected exercise heart rate ranges.
- **`oldpeak`** (ST depression): Heavily right-skewed with a large spike
  at 0 — most patients show no ST depression, with a smaller group
  showing moderate to high values (up to ~6). A few negative values are
  visible, which is unusual and worth double-checking.
- **`ca`** (number of major vessels colored by fluoroscopy): Not
  continuous — it takes only 4 discrete values (0, 1, 2, 3), with 0 being
  by far the most common. This is more like an ordinal/categorical
  variable despite being stored as numeric.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    df[col].value_counts(dropna=False).plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Categorical Feature Distributions

**Demographics & Data Source:**
- **`sex`**: Strongly imbalanced — roughly 720 male vs. 190 female
  patients (~79% male). This imbalance should be kept in mind, as model
  performance may be less reliable for the underrepresented female group.
- **`dataset`**: Data comes from 4 hospitals (Cleveland, Hungary, VA Long
  Beach, Switzerland), with Cleveland and Hungary contributing the most
  records. Since data-quality varies by source (seen below), this column
  is useful for tracing where missing values originate.

**Clinical Indicators:**
- **`cp` (chest pain type)**: Most patients are asymptomatic (~500),
  which is somewhat counterintuitive — it reflects that this label means
  "no chest pain symptom reported," not "no disease."
- **`fbs` (fasting blood sugar)**: Mostly False (~700), meaning most
  patients do not have elevated fasting blood sugar. A small number of
  missing values (`nan`) are present.
- **`restecg`**: Mostly "normal" (~550), with meaningful numbers of "lv
  hypertrophy" and "st-t abnormality." Almost no missing values here.
- **`exang` (exercise-induced angina)**: Mostly False (~530), with
  moderate True cases (~340) — a reasonably balanced signal.
- **`slope`**: Roughly 300+ missing values (`nan`) — a substantial
  data-quality issue that will need to be addressed before modeling.
- **`thal`**: The **largest missing-value problem** in the dataset —
  "nan" is actually the *most frequent* category (~490 records),
  outnumbering all real categories combined. This column will need
  careful handling (imputation or exclusion).

**Target Variable (`num`):**
- Clearly **imbalanced across severity levels**: ~410 patients have no
  disease (0), while severity levels 1-4 progressively shrink (1: ~260,
  2: ~110, 3: ~110, 4: ~30). If treated as multiclass, levels 3 and 4 have
  very few samples — likely too few for reliable classification. This
  supports converting `num` into a **binary target** (0 = no disease,
  1+ = disease present) for a more robust classification problem.

In [ ]:
df.isnull().sum()

In [ ]:
cols_to_impute = ['trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak','ca', 'thal', 'slope']

numeric_cols = df[cols_to_impute].select_dtypes(include='number').columns.tolist()
categorical_cols = df[cols_to_impute].select_dtypes(exclude='number').columns.tolist()

In [ ]:
# Preprocessing for numeric columns: impute missing values with median
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Preprocessing for categorical columns: impute with mode, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

In [ ]:
df.isnull().sum()

### Handling Missing Values

Missing values ranged widely across columns — from none in `age`, `sex`,
`dataset`, `cp`, and `num`, to just 2 in `restecg`, up to 611 (~66%) in
`ca`. Rather than dropping any columns, every column was imputed based on
its type: numeric columns (`trestbps`, `chol`, `fbs`, `thalch`, `exang`,
`oldpeak`, `slope`, `ca`) were filled with their median, robust against
outliers, while the categorical column `restecg` was filled with its mode,
since a median isn't meaningful for text categories like `normal` or
`lv hypertrophy`.

**Result:** all missing values are now filled, with the full dataset
preserved.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col])
    plt.title(f"Box Plot: {col}")
    plt.show()

### Box Plot Findings: Numeric Columns

Scanning across all numeric columns reveals a clear pattern: most
distributions carry **outliers on the high end**, and one column shows a
value that isn't just an outlier — it's likely a data error.

- **`age`:** clean, no outliers.
- **`trestbps` (resting blood pressure):** several high outliers above
  ~170, plus **one value near 0** — a resting blood pressure of 0 isn't
  physiologically possible, so this is almost certainly a data-entry error
  rather than a genuine rare case.
- **`chol` (cholesterol):** a cluster of high outliers above ~400, and
  also **one value near 0**, the same likely error pattern as `trestbps`.
- **`thalch` (max heart rate):** a couple of low outliers near 60-70,
  otherwise clean — plausible for a genuinely low heart rate reading.
- **`oldpeak`:** several high outliers above ~4, plus one negative value —
  worth checking, since `oldpeak` (ST depression) is typically ≥0.
- **`ca`:** appears almost entirely one value (0), with a few points at
  1, 2, and 3 flagged as "outliers" — this is likely a discrete/count
  variable rather than continuous, so the IQR method isn't the right tool
  here.
- **`num`:** no outliers; this looks like the target variable (disease
  severity, 0-4), and its spread is expected, not anomalous.
  the zero values in `trestbps` and `chol` are the
real issue — they're implausible for a living patient and should be
treated as missing/invalid rather than genuine extreme values, then
handled the same way other missing data was (e.g. median imputation)
rather than left in as if they were real measurements.

In [ ]:
# Delete zero rows with trestbps or chol (obvious data errors)
df = df[(df['trestbps'] != 0) & (df['chol'] != 0)]

print(df.shape)

In [ ]:
# Examine the existing category columns first.
categorical_cols = df.select_dtypes(exclude='number').columns.tolist()
print("Categorical columns:", categorical_cols)

for col in categorical_cols:
    print(f"\n{col}: {df[col].unique()}")

Before encoding, we need to see exactly what text columns are present and what the unique values ​​are in each unit — because some columns (like `sex` or `cp`) may be simple binary, while other columns (`restecg`, `dataset`) have more than two classes and require one-hot encoding.

In [ ]:
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df.shape)
print(df.dtypes)

The categorical (text) columns were converted to digital columns using one-hot encoding, with `drop_first=True` to avoid redundancy between categories (multicollinearity). This made every column in the dataset fully digital and ready for immediate use in any machine learning model.

In [ ]:
corr = df.corr()  # corr = df.corr() # All columns are numeric by default (including bool = 0/1)
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix — All Features")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Use df_raw: data after dropping id and after removing trestbps/chol=0 rows only
# (before manual fillna, before get_dummies, before manual feature engineering)
X = df.drop('num', axis=1)
y = (df['num'] > 0).astype(int)   # binary instead of 5 classes

# Step 1: 60% training, 40% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Step 2: split the 40% into val/test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)

# 5-fold cross-validation with stratification (appropriate since num has multiple classes)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=skf,
    scoring='f1_macro'   # appropriate for multi-class classification
)

print("F1 per fold:", cv_scores)
print(f"\nMean F1: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

5-fold cross-validation was applied using `StratifiedKFold` instead of
plain `KFold`, since `num` has multiple classes (0-4) and some are likely
rarer than others — stratification ensures every fold preserves the same
class distribution present in the overall data, rather than letting folds
end up with imbalanced, random class splits.

`f1_macro` was used instead of `accuracy`, because in multi-class problems
— especially with rare classes — accuracy alone can be just as misleading
as it was in binary classification, while `f1_macro` weighs every class
equally instead of favoring the most frequent one.

### Cross-Validation Results

Mean F1 (macro) = **0.7898**, SD = **0.0273** — fold scores are close
together, so this is a stable estimate, not a lucky/unlucky split.

The low mean itself is the concern: ~0.30 F1 on a 5-class target (`num` =
0-4) suggests the model struggles, likely due to class imbalance — rare
severity classes get penalized heavily by `f1_macro`.

In [ ]:
print(y.value_counts().sort_index())
print("\nPercentage:")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

### Class Distribution: `num`

The imbalance is confirmed and severe: class 0 (no disease) makes up
**52%** of the data, class 1 makes up **27%**, but classes 2, 3, and 4
combined are only **20%** — with class 4 at just **2.94%** (22 patients
total). This explains the low F1 macro score directly: with so few
examples of classes 2-4, the model has almost nothing to learn their
patterns from, and `f1_macro` penalizes that poor minority-class
performance heavily since it weighs all 5 classes equally regardless of size.

In [ ]:
# Reframe num as binary: 0 = no disease, 1 = disease present (any severity)
y_binary = (y > 0).astype(int)

print(y_binary.value_counts())
print((y_binary.value_counts(normalize=True) * 100).round(2))

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Re-split using the binary target
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_binary, test_size=0.4, random_state=42, stratify=y_binary
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Re-run cross-validation on the training set
model = RandomForestClassifier(n_estimators=100, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model, X_train, y_train,
    cv=skf,
    scoring='f1'   # binary F1 now, not f1_macro
)

print("F1 per fold:", cv_scores)
print(f"\nMean F1: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

### Cross-Validation Results: Binary Classification

Mean F1 jumped from **0.3029** (5-class) to **0.7881** (binary) — a
**160% improvement**. The standard deviation (0.0326) is close to before,
confirming this is a stable estimate, not a lucky split.

This confirms the diagnosis was correct: **the low multi-class score was
caused by class imbalance across 5 sparse severity levels**, not by weak
features or a poor model choice. Once reframed as a binary question
("disease present or not"), the same features and same Random Forest
model perform strongly — catching genuine disease cases with reasonably
high, consistent F1 across all 5 folds.

In [ ]:
# Train the final model on the full training set
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate on the test set — ONE TIME ONLY
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

y_pred = final_model.predict(X_test)

print("Test F1:", f1_score(y_test, y_pred))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.show()

#### Final Test Set Evaluation

Test F1 = **0.76**, Accuracy = **0.78** — below the CV mean of 0.7881,
within normal single-split variance.

**Confusion Matrix:**
| | Predicted: No | Predicted: Yes |
|---|---|---|
| **Actual: No** | 67 (TN) | 12 (FP) |
| **Actual: Yes** | 20 (FN) | 51 (TP) |

**20 false negatives** — the costliest error type for a screening context.
Recall on the disease class is only **0.68**, lower than its precision
(0.75).

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['hr_reserve'] = (220 - X['age']) - X['thalch']
        X['bp_chol_product'] = X['trestbps'] * X['chol']
        X['high_risk_flag'] = (
            (X['exang'] == True) & (X['oldpeak'] > 1.0) & (X['ca'] > 0)
        ).astype(int)
        return X

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [ ]:
print(X_train.isnull().sum())

In [ ]:
cols_with_nan = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']

for col in cols_with_nan:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)
    X_val[col] = X_val[col].fillna(median_val)

print("Remaining NaN in X_train:", X_train.isnull().sum().sum())
print("Remaining NaN in X_test:", X_test.isnull().sum().sum())
print("Remaining NaN in X_val:", X_val.isnull().sum().sum())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
for name, clf in models.items():
    scores = cross_val_score(clf, X_train, y_train, cv=skf, scoring='f1')
    results.append({'Model': name, 'Mean F1': scores.mean(), 'Std': scores.std()})

results_df = pd.DataFrame(results).sort_values('Mean F1', ascending=False)
print(results_df)

### Model Comparison: Cross-Validated F1

| Model | Mean F1 | Std |
|---|---|---|
| **Logistic Regression** | **0.793284** | 0.035 |
| Random Forest (current) | 0.7851 | 0.031 |

**Logistic Regression wins** — highest F1, beating Random Forest by ~1.7
points, suggesting the relationship is largely linear.

**Recommendation:** switch to Logistic Regression — more accurate and
more interpretable for a medical context.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced', None]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

### GridSearchCV Results

**Best Parameters:** `C=1`, `penalty='l2'`, `class_weight='balanced'`
**Best CV F1: 0.8115** — a small improvement over the untuned default
(0.8046), a gain of about 0.7 points.

The tuning confirms `class_weight='balanced'` helps — exactly the fix
targeted at the false negative problem seen earlier. `C=1` (a moderate,
default-like regularization strength) winning suggests the default was
already close to optimal; the search mainly validated the setup rather
than finding a dramatically better configuration.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'C': np.logspace(-3, 2, 50),   # 50 values ​​distributed logarithmically between 0.001 and 100
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced', None]
}

random_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_distributions=param_dist,
    n_iter=30,              # He only tries 30 random combinations, not all possibilities.
    cv=skf,
    scoring='f1',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV F1:", random_search.best_score_)

In [ ]:
final_model = LogisticRegression(
    C=1, penalty='l2', class_weight='balanced',
    solver='liblinear', max_iter=1000, random_state=42
)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)

print("Test F1:", f1_score(y_test, y_pred))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.show()

### Final Model: Random Forest vs. Tuned Logistic Regression

| Metric | Random Forest | Tuned LogReg | Change |
|---|---|---|---|
| Test F1 | 0.7111 | **0.7660** | +0.055 |
| Accuracy | 0.74 | **0.78** | +0.04 |
| False Negatives | 23 | **17** | −6 |
| Recall (disease) | 0.68 | **0.76** | +0.08 |

`class_weight='balanced'` worked as intended — 6 more real disease cases
caught, with no drop in precision.



In [ ]:
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': final_model.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(coefficients)

### Feature Coefficients: What Drives the Prediction

**Increases risk:**
- `sex_Male` (+1.10) — known risk factor
- `exang_True` (+0.91) — exercise-induced angina, classic cardiac symptom
- `ca` (+0.89) — more affected vessels, higher risk
- `slope_flat` (+0.58), `oldpeak` (+0.52) — ST-segment stress test indicators
- `dataset_VA Long Beach` (+0.69) — likely a site/population effect, not clinical

**Decreases risk:**
- `cp_atypical angina` (−1.52) — strongest effect overall; non-asymptomatic
  chest pain types are protective vs. the asymptomatic baseline
- `cp_non-anginal` (−0.90), `cp_typical angina` (−0.78)
- `thal_normal` (−0.69) — normal test result, as expected

**Negligible:** `age`, `trestbps`, `chol`, `thalch` — all near zero,
likely because their signal is captured indirectly through `cp`, `exang`, `ca`.

In [ ]:
from sklearn.metrics import f1_score

# Model performance on the same training data
y_train_pred = final_model.predict(X_train)
train_f1 = f1_score(y_train, y_train_pred)

# Model performance on test data (never seen before)
y_test_pred = final_model.predict(X_test)
test_f1 = f1_score(y_test, y_test_pred)

print(f"Train F1: {train_f1:.4f}")
print(f"Test F1: {test_f1:.4f}")
print(f"Gap: {train_f1 - test_f1:.4f}")

In [ ]:
df.head()

In [ ]:
df


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Separate features from target
X = df.drop(columns=['num'])

# Impute remaining missing values
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Apply K-Means
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

# Show cluster counts
print("Number of clusters:", df['KMeans_Cluster'].nunique())
print("\nCluster sizes:")
print(df['KMeans_Cluster'].value_counts().sort_index())

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['KMeans_Cluster'],
    s=50
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clustering')
plt.colorbar(scatter, label='Cluster')
plt.show()

#### K-Means Clustering

K-Means was applied to the preprocessed medical dataset after handling the remaining missing values, removing the target variable `num`, and scaling the features.

The number of clusters was set to **4**.

PCA was then used to reduce the features to two principal components (`PC1` and `PC2`) for visualization of the resulting clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

inertias = []
silhouette_scores = []

for k in range(2, 11):
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = km.fit_predict(X_scaled)

    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Elbow plot
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(range(2, 11))
plt.show()

# Silhouette plot
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score')
plt.xticks(range(2, 11))
plt.show()

K=4 looks like the best balance: it's where the elbow plot shows the first big bend in inertia, and where silhouette score jumps up from its low baseline (k=2,3 ~0.13 → k=4 ~0.153).

If you only care about silhouette score, k=8 or k=10 score higher (~0.165–0.17), but overall scores are low across the board (<0.2), suggesting weak cluster separation regardless of k — worth checking if features are scaled.

Avoid k=7 (local dip in silhouette despite inertia still decreasing).

In [ ]:
from sklearn.cluster import DBSCAN

eps_value = 1.2  # replace with the value suggested by the plot

dbscan = DBSCAN(
    eps=eps_value,
    min_samples=5
)

dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = (dbscan_labels == -1).sum()

print("DBSCAN Results")
print("----------------")
print("eps:", eps_value)
print("Number of clusters:", n_clusters)
print("Number of noise points:", n_noise)

#### DBSCAN Clustering Results

DBSCAN was applied to the same scaled dataset using `eps = 1.2` and `min_samples = 5`.

- **Number of clusters:** 9
- **Number of noise points:** 661

DBSCAN identified **9 clusters**, while **661 observations** were classified as noise because they did not belong to sufficiently dense regions.

In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot DBSCAN clusters
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=dbscan_labels,
    s=50
)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('DBSCAN Clustering (eps=1.2)')
plt.colorbar(scatter, label='Cluster')
plt.show()

In [ ]:

for eps in [0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 2.0]:

    dbscan = DBSCAN(
        eps=eps,
        min_samples=5
    )

    labels = dbscan.fit_predict(X_scaled)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    print(
        f"eps={eps}: "
        f"clusters={n_clusters}, "
        f"noise={n_noise}"
    )

#### DBSCAN Parameter Selection

Different `eps` values were tested while keeping `min_samples = 5` to observe their effect on the number of clusters and noise points.

| `eps` | Clusters | Noise Points |
|---:|---:|---:|
| 0.5 | 0 | 748 |
| 0.7 | 1 | 743 |
| 0.9 | 2 | 738 |
| 1.0 | 4 | 719 |
| 1.2 | 9 | 661 |
| 1.5 | 10 | 597 |
| 2.0 | 19 | 510 |

As `eps` increases, more points become connected, resulting in more clusters and fewer noise points.

For this experiment, `eps = 1.2` was selected, producing **9 clusters and 661 noise points**.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Build hierarchical clustering
Z = linkage(X_scaled, method='ward')

# Plot dendrogram
plt.figure(figsize=(12, 6))

dendrogram(Z)

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Data Points')
plt.ylabel('Distance')

plt.axhline(
    y=10,
    linestyle='--',
    label='Cut Height = 10'
)

plt.legend()
plt.show()

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

In [ ]:
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print("Number of components:", n_components_95)
print("Explained variance:", cumulative_variance[n_components_95 - 1])

#### PCA Component Selection

We selected **16 principal components** because they retain approximately **95.20% of the total variance**.

This provides a good balance between **dimensionality reduction** and **preserving the information** in the original dataset. Using 16 components reduces the number of features while maintaining approximately 95% of the dataset's variability.

In [ ]:
pca_95 = PCA(n_components=n_components_95)

X_pca_95 = pca_95.fit_transform(X_scaled)

print("Original shape:", X_scaled.shape)
print("Reduced shape:", X_pca_95.shape)

#### PCA Dimensionality Reduction

The original dataset contained **20 features** and **748 samples**.

After applying PCA with **16 components**, the dataset was reduced to **16 features** while retaining approximately **95.20% of the total variance**.

- **Original shape:** `(748, 20)`
- **Reduced shape:** `(748, 16)`
- **Variance retained:** `95.20%`

This reduces the dimensionality while preserving most of the information in the original dataset.

In [ ]:
# Reduce the scaled data to 2 principal components
pca_2d = PCA(n_components=2)

X_pca_2d = pca_2d.fit_transform(X_scaled)

print("Reduced shape:", X_pca_2d.shape)

#### 2D PCA Reduction

The scaled dataset was reduced from **20 features to 2 principal components** using PCA.

- **Original shape:** `(748, 20)`
- **Reduced shape:** `(748, 2)`

Each sample is now represented by two principal components, **PC1** and **PC2**, which can be visualized in a 2D scatter plot.

In [ ]:
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    X_pca_2d[:, 0],
    X_pca_2d[:, 1],
    c=y,
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("2D PCA Projection")
plt.colorbar(scatter, label="Group")

plt.show()